In [1]:
! pip install -q torch torchvision faiss-cpu transformers chromadb pillow


In [2]:
# Install dependencies:

import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import chromadb
from chromadb.config import Settings
import numpy as np

# Load CLIP
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

# Ingest and embed images
def embed_image(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        image_emb = model.get_image_features(**inputs)
    return image_emb.squeeze().cpu().numpy()

# (For text queries)
def embed_text(query):
    inputs = processor(text=[query], return_tensors="pt")
    with torch.no_grad():
        text_emb = model.get_text_features(**inputs)
    return text_emb.squeeze().cpu().numpy()

# Store in a vector DB (Chroma for example)
chroma_client = chromadb.Client(Settings())
collection = chroma_client.create_collection(name="image_embeddings")

# Index images
image_paths = ["image1.jpg", "image2.jpg", "image3.jpg"]
for idx, path in enumerate(image_paths):
    emb = embed_image(path)
    collection.add(embeddings=[emb], ids=[str(idx)], metadatas=[{"path": path}])

# Retrieval: Given a query, find most similar images
def retrieve_images(query, top_k=2):
    query_emb = embed_text(query)
    results = collection.query(query_embeddings=[query_emb], n_results=top_k)
    return [collection.get(id)["metadata"]["path"] for id in results["ids"][0]]

# Example usage
query = "A mountain with snow"
top_imgs = retrieve_images(query, top_k=2)
print(f"Top matched image files: {top_imgs}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/suljain/opt/anaconda3/envs/rag_env/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/suljain/opt/anaconda3/envs/rag_env/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/suljain/opt/anaconda3/envs/rag_env/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 7

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434